# BGE Reranker Evaluation on MTEB/SciFact
Run this notebook on Kaggle or Google Colab with a GPU enabled (T4 or P100) to evaluate the BAAI/bge-reranker-v2-m3 model in seconds.

In [ ]:
!pip install -q sentence-transformers faiss-gpu datasets numpy

In [ ]:
import re
import numpy as np
from math import log
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

# --- METRICS ---
def precision_at_k(relevant, retrieved, k):
    hits = sum(1 for d in retrieved[:k] if d in relevant)
    return hits / k if k > 0 else 0.0

def mrr(relevant, retrieved):
    for rank, d in enumerate(retrieved, 1):
        if d in relevant: return 1.0 / rank
    return 0.0

def dcg_at_k(relevant, retrieved, k):
    return sum(1.0 / np.log2(i + 2) for i, d in enumerate(retrieved[:k]) if d in relevant)

def ndcg_at_k(relevant, retrieved, k):
    dcg = dcg_at_k(relevant, retrieved, k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(k, len(relevant))))
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(relevant, retrieved, k):
    if not relevant: return 0.0
    hits = len(set(retrieved[:k]).intersection(set(relevant)))
    return hits / len(relevant)

def average_precision(relevant, retrieved):
    if not relevant: return 0.0
    hits = 0
    sum_prec = 0.0
    rel_set = set(relevant)
    for rank, d in enumerate(retrieved, 1):
        if d in rel_set:
            hits += 1
            sum_prec += hits / rank
    return sum_prec / len(relevant)

In [ ]:
# --- RETRIEVAL CLASSES ---
class BM25Index:
    def __init__(self):
        self.df = {}
        self.idf = {}
        self.doc_term_freqs = []
        self.doc_lengths = []
        self.avg_doc_length = 0.0
        self.n_docs = 0
        self.k1 = 1.5
        self.b = 0.75

    def fit(self, docs):
        self.n_docs = len(docs)
        for doc in docs:
            tokens = [t for t in re.findall(r"[a-zA-Z0-9]+", doc.lower()) if len(t) >= 2]
            self.doc_lengths.append(len(tokens))
            tf = {}
            for tok in tokens:
                tf[tok] = tf.get(tok, 0) + 1
            self.doc_term_freqs.append(tf)
            for tok in set(tokens):
                self.df[tok] = self.df.get(tok, 0) + 1
        self.avg_doc_length = sum(self.doc_lengths) / max(1, self.n_docs)
        for term, df in self.df.items():
            self.idf[term] = log((self.n_docs - df + 0.5) / (df + 0.5) + 1.0)

    def score(self, query, top_k=100):
        tokens = [t for t in re.findall(r"[a-zA-Z0-9]+", query.lower()) if len(t) >= 2]
        scores = np.zeros(self.n_docs, dtype=np.float64)
        for term in tokens:
            if term not in self.idf: continue
            idf = self.idf[term]
            for doc_idx in range(self.n_docs):
                tf = self.doc_term_freqs[doc_idx].get(term, 0)
                if tf == 0: continue
                doc_len = self.doc_lengths[doc_idx]
                num = tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avg_doc_length)
                scores[doc_idx] += idf * num / den
        top_idx = np.argsort(scores)[::-1][:top_k]
        return scores[top_idx], top_idx

class HybridSearcher:
    def __init__(self, bm25, alpha=0.5):
        self.bm25 = bm25
        self.alpha = alpha

    def search(self, query, q_emb, faiss_idx, top_k=100):
        n_cand = top_k * 3
        d_dists, d_idx = faiss_idx.search(np.array([q_emb]), n_cand)
        d_dists, d_idx = d_dists[0], d_idx[0]
        
        s_scores, s_idx = self.bm25.score(query, n_cand)
        
        candidates = set([int(x) for x in d_idx if x >= 0] + [int(x) for x in s_idx if x >= 0])
        
        dense_map = {int(idx): float((dist+1)/2) for dist, idx in zip(d_dists, d_idx) if idx >= 0}
        bm25_max = max(s_scores) if len(s_scores) > 0 and max(s_scores) > 0 else 1.0
        bm25_min = min(s_scores) if len(s_scores) > 0 else 0.0
        bm_range = bm25_max - bm25_min if bm25_max > bm25_min else 1.0
        sparse_map = {int(idx): float((score-bm25_min)/bm_range) for score, idx in zip(s_scores, s_idx) if idx >= 0}
        
        scored = []
        for idx in candidates:
            hybrid = self.alpha * dense_map.get(idx, 0.0) + (1 - self.alpha) * sparse_map.get(idx, 0.0)
            scored.append({"idx": idx, "score": hybrid})
            
        scored.sort(key=lambda x: x["score"], reverse=True)
        top = scored[:top_k]
        return [x["idx"] for x in top], [x["score"] for x in top]

In [ ]:
# --- LOAD DATA ---
print("Loading Dataset...")
corpus = load_dataset("mteb/scifact", "corpus", split="corpus")
docs = [f"{item['title']} {item['text']}".strip() for item in corpus]
doc_id_to_idx = {item["_id"]: i for i, item in enumerate(corpus)}

queries = load_dataset("mteb/scifact", "queries", split="queries")
q_id_to_text = {item["_id"]: item["text"] for item in queries}

qrels = load_dataset("mteb/scifact", "default", split="test")
from collections import defaultdict
benchmark = defaultdict(list)
for item in qrels:
    if item["score"] > 0 and item["corpus-id"] in doc_id_to_idx:
        benchmark[item["query-id"]].append(doc_id_to_idx[item["corpus-id"]])

eval_queries = [{"query": q_id_to_text[q], "relevant": rel} for q, rel in benchmark.items() if q in q_id_to_text][:50]
print(f"Loaded {len(eval_queries)} queries with ground truth.")

In [ ]:
# --- BUILD INDICES ---
print("Building BM25...")
bm25 = BM25Index()
bm25.fit(docs)

print("Building FAISS Index (Dense)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embs = embedder.encode(docs, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
faiss_idx = faiss.IndexFlatIP(384)
faiss_idx.add(doc_embs)

hybrid = HybridSearcher(bm25, alpha=0.5)

In [ ]:
# --- LOAD BGE RERANKER ---
print("Loading BAAI/bge-reranker-v2-m3...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

In [ ]:
# --- EVALUATE ---
metrics = {
    "Hybrid": {"p@3": [], "mrr": [], "ndcg@10": [], "recall@10": [], "map": []},
    "Reranked": {"p@3": [], "mrr": [], "ndcg@10": [], "recall@10": [], "map": []}
}

print("Evaluating...")
for i, item in enumerate(eval_queries):
    print(f"Query {i+1}/{len(eval_queries)}", end="\r")
    q_text, rel = item["query"], item["relevant"]
    q_emb = embedder.encode([q_text], normalize_embeddings=True)[0]
    
    # Hybrid
    h_idx, _ = hybrid.search(q_text, q_emb, faiss_idx, top_k=100)
    h_idx_10 = h_idx[:10]
    metrics["Hybrid"]["p@3"].append(precision_at_k(rel, h_idx_10, 3))
    metrics["Hybrid"]["mrr"].append(mrr(rel, h_idx_10))
    metrics["Hybrid"]["ndcg@10"].append(ndcg_at_k(rel, h_idx_10, 10))
    metrics["Hybrid"]["recall@10"].append(recall_at_k(rel, h_idx_10, 10))
    metrics["Hybrid"]["map"].append(average_precision(rel, h_idx_10))
    
    # Reranker
    cand_docs = [docs[idx] for idx in h_idx]
    pairs = [[q_text, d] for d in cand_docs]
    scores = reranker.predict(pairs)
    scored = sorted(zip(h_idx, scores), key=lambda x: x[1], reverse=True)
    r_idx_10 = [x[0] for x in scored[:10]]
    
    metrics["Reranked"]["p@3"].append(precision_at_k(rel, r_idx_10, 3))
    metrics["Reranked"]["mrr"].append(mrr(rel, r_idx_10))
    metrics["Reranked"]["ndcg@10"].append(ndcg_at_k(rel, r_idx_10, 10))
    metrics["Reranked"]["recall@10"].append(recall_at_k(rel, r_idx_10, 10))
    metrics["Reranked"]["map"].append(average_precision(rel, r_idx_10))

print("\n\nResults:")
print(f"{str('Mode'):<10} | {'P@3':<8} | {'MRR':<8} | {'NDCG':<8} | {'R@10':<8} | {'MAP':<8}")
for mode in ["Hybrid", "Reranked"]:
    p3 = np.mean(metrics[mode]["p@3"])
    mrr_v = np.mean(metrics[mode]["mrr"])
    ndcg = np.mean(metrics[mode]["ndcg@10"])
    rec = np.mean(metrics[mode]["recall@10"])
    map_v = np.mean(metrics[mode]["map"])
    print(f"{mode:<10} | {p3:.4f}   | {mrr_v:.4f}   | {ndcg:.4f}   | {rec:.4f}   | {map_v:.4f}")